# Differentiable VerCOR + Veros-AD: gradients through a coupled rollout

This notebook demonstrates `jax` autodiff through the coupled ATM (ERA5 forcing) + OCN (Veros GCM) + LND (ERA5 forcing) system built in `examples/run_verosad_global4deg_grad.py`:

1. **Forward integration** -- run the coupled system forward and look at how a couple of ocean fields evolve.
2. **Spatial gradient** -- d(loss)/d(initial temperature field), one value per grid cell.
3. **Scalar gradient** -- d(loss)/d(K_gm_0), the ocean's fixed eddy (Gent-McWilliams) diffusivity.

Component setup, rollout bookkeeping, and plotting live in `_verosad_grad_helpers.py` next to this notebook -- this notebook only shows the `jax` calls.

In [ ]:
from datetime import datetime

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from _verosad_grad_helpers import (
    build_coupler,
    disable_eke,
    ocean_payload,
    plot_field_evolution,
    plot_gradient_map,
    rollout_and_capture,
    set_ocean_variable,
)

START = datetime(2000, 1, 1)
ROLLOUT_STEPS = 2  # days -- kept short so the notebook runs quickly

## 1. Forward integration: field evolution

Roll the coupled system forward a few days and look at how two ocean surface fields (temperature, salinity) evolve. Vercor's built-in per-day NetCDF output does the bookkeeping here -- see `rollout_and_capture` / `plot_field_evolution` in the helper file.

In [ ]:
initial_state, output_dir = rollout_and_capture(START, total_steps=6, variables=("temp", "salt"))

plot_field_evolution(initial_state, output_dir, "temp", every=2, label="surface temperature (deg C)")
plot_field_evolution(initial_state, output_dir, "salt", every=2, label="surface salinity (g/kg)")
plt.show()

## 2. Spatial gradient: d(loss)/d(initial temperature)

Differentiate a scalar loss computed after a short rollout with respect to the *full* initial 3D temperature field -- one gradient value per grid cell, showing where the initial ocean temperature most influences the loss. The loss here is simply sum(final_temp ** 2).

In [ ]:
cpl = build_coupler(START, ROLLOUT_STEPS)
base_state = cpl.initial_state()
initial_temp = ocean_payload(base_state).variables.temp


def loss_fn_temp(temp_field):
    state = set_ocean_variable(base_state, "temp", temp_field)
    result = cpl.run(state, output=None)
    return jnp.sum(ocean_payload(result).variables.temp ** 2)


loss, grad_temp = jax.value_and_grad(loss_fn_temp)(initial_temp)

print(f"loss                 = {float(loss):.6e}")
print(f"NaN in grad          = {int(jnp.isnan(grad_temp).sum())} / {grad_temp.size}")
print(f"grad abs max / mean  = {float(jnp.abs(grad_temp).max()):.3e} / {float(jnp.abs(grad_temp).mean()):.3e}")

plot_gradient_map(ocean_payload(base_state), grad_temp, title="d(loss)/d(initial temperature), surface")
plt.show()

## 3. Scalar gradient: d(loss)/d(K_gm_0)

`K_gm_0` is Veros' fixed Gent-McWilliams eddy diffusivity -- it only drives the model when the EKE parameterization is switched off (`disable_eke`). Same recipe as above, but with a scalar parameter instead of a field, checked against a central finite difference (as in `Veros-Autodiff/notebooks/demonstration/gradient-computation.ipynb`).

In [ ]:
K_GM_0 = jnp.asarray(1000.0)  # typical GM diffusivity, m^2/s

cpl = build_coupler(START, ROLLOUT_STEPS)
base_state = disable_eke(cpl.initial_state())


def loss_fn_kgm(k_gm_0):
    state = set_ocean_variable(base_state, "K_gm_0", k_gm_0)
    result = cpl.run(state, output=None)
    return jnp.sum(ocean_payload(result).variables.temp ** 2)


value, grad = jax.value_and_grad(loss_fn_kgm)(K_GM_0)

eps = 1.0
fd_grad = (loss_fn_kgm(K_GM_0 + eps) - loss_fn_kgm(K_GM_0 - eps)) / (2 * eps)
rel_error = abs(grad - fd_grad) / abs(fd_grad)

print(f"loss                           = {float(value):.6e}")
print(f"d(loss)/d(K_gm_0)  autodiff    = {float(grad):.6e}")
print(f"d(loss)/d(K_gm_0)  finite diff = {float(fd_grad):.6e}")
print(f"relative error                 = {float(rel_error):.3e}")